In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
import pandas as pd
import time
import os

In [2]:
def checkNewsDateispresent_or_not(savepath):
    saved_date = None
    old_html_rows = ""
    
    if os.path.exists(savepath):
        old_df = pd.read_html(savepath, attrs={"id": "myTableCNews"})[0]
        saved_date = pd.to_datetime(old_df.iloc[0, 0])
        
        with open(savepath, "r", encoding="utf-8") as f:
            old_content = f.read()
            
        old_html_rows = old_content.split("<tbody>")[1].split("</tbody>")[0]
        
    return saved_date, old_html_rows


filepath = "../DATA-HTML-STOCK/NEPSECompany/NEPSECompanyExtractor.csv"
df = pd.read_csv(filepath)
symbols = df['Symbol'].tolist()

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(service=Service("/usr/bin/chromedriver"), options=options)
driver.set_page_load_timeout(30)

print(f"Found {len(symbols)} companies to scrape")

for index,symbol in enumerate(symbols):
    
    print(f"\nScraping {symbol}...")
    
    savepath = f"../DATA-HTML-STOCK/webScrapped-htmlfiles/news-WEB-SCRAP-htmlfile/{symbol}news.html"
    saved_date, old_html_rows = checkNewsDateispresent_or_not(savepath)
    
    try:
        driver.get(f'https://www.sharesansar.com/company/{symbol}')
        time.sleep(3)
        
        driver.find_elements(By.LINK_TEXT, "News")[1].click()
        time.sleep(3)
        
        all_new_rows = []
        page = 1
        stop_scraping = False
        
        while True:
            rows = driver.find_elements(By.CSS_SELECTOR, "#myTableCNews tbody tr")
            
            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                scraped_date = pd.to_datetime(cols[0].text)
                
                if saved_date is None:
                    all_new_rows.append(row.get_attribute("outerHTML"))
                else:
                    if scraped_date > saved_date:
                        all_new_rows.append(row.get_attribute("outerHTML"))
                    else:
                        stop_scraping = True
                        break
                        
            if stop_scraping:
                break
                
            next_button = driver.find_element(By.ID, "myTableCNews_next")
            if "disabled" in next_button.get_attribute("class"):
                break
                
            next_button.click()
            print(f"✓ page: {page} ", end=" ")
            time.sleep(2)
            page += 1
            
        if saved_date is None:
            final_rows = ''.join(all_new_rows)
        else:
            final_rows = ''.join(all_new_rows) + old_html_rows
            
        if final_rows.strip():
            html_content = f"""<table id="myTableCNews">
<tbody>
{final_rows}
</tbody>
</table>"""
            os.makedirs(os.path.dirname(savepath), exist_ok=True)

            with open(savepath, "w", encoding="utf-8") as f:
                f.write(html_content)

            print(f"Saved {len(all_new_rows)} new rows")
            print(f"{len(symbols)-(index+1)} remain to scrap\n")
        else:
            print("No new data found")
    except TimeoutException:
        print("Page load timeout")
    except Exception as e:
        print(f"Error scraping {symbol}: {e}")

driver.quit()
print("All done!")

Found 300 companies to scrape

Scraping API...
✓ page: 1  ✓ page: 2  ✓ page: 3  ✓ page: 4  ✓ page: 5  ✓ page: 6  ✓ page: 7  ✓ page: 8  ✓ page: 9  ✓ page: 10  ✓ page: 11  ✓ page: 12  ✓ page: 13  ✓ page: 14  ✓ page: 15  ✓ page: 16  ✓ page: 17  ✓ page: 18  ✓ page: 19  ✓ page: 20  ✓ page: 21  ✓ page: 22  ✓ page: 23  Saved 232 new rows
299 remain to scrap


Scraping HATH...
✓ page: 1  ✓ page: 2  ✓ page: 3  ✓ page: 4  ✓ page: 5  ✓ page: 6  ✓ page: 7  Saved 71 new rows
298 remain to scrap


Scraping HATHPO...
Error scraping HATHPO: Unknown datetime string format, unable to parse: No data available in table

Scraping AKPL...
✓ page: 1  ✓ page: 2  ✓ page: 3  ✓ page: 4  ✓ page: 5  ✓ page: 6  ✓ page: 7  ✓ page: 8  ✓ page: 9  ✓ page: 10  ✓ page: 11  Saved 113 new rows
296 remain to scrap


Scraping AHPC...
✓ page: 1  ✓ page: 2  ✓ page: 3  ✓ page: 4  ✓ page: 5  ✓ page: 6  ✓ page: 7  ✓ page: 8  ✓ page: 9  ✓ page: 10  ✓ page: 11  ✓ page: 12  ✓ page: 13  ✓ page: 14  ✓ page: 15  ✓ page: 16  ✓ page: 17 